# QLoRA fine-tune: distill the teacher into a local student

**Story:** the OpenRouter teacher (DeepSeek V3.2) scores 100% on the golden
set. This notebook trains a 1.5B open model on template-verified Q→SQL pairs
(`data/train.jsonl`, generated by `src/gen_training_data.py`) and measures
how close the student gets — at $0/query.

**Runtime:** free Colab T4 (~15GB) is enough for Qwen2.5-Coder-1.5B QLoRA.

Upload `data/train.jsonl` to the session before running cell 3.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes datasets

In [ ]:
from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit"  # 4-bit base, fits T4
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=2048,   # schema + glossary + question fits easily
    load_in_4bit=True,     # QLoRA: quantized base + LoRA adapters
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
from datasets import load_dataset

# train.jsonl: {"messages": [system, user, assistant]} — mirrors inference
ds = load_dataset("json", data_files="train.jsonl", split="train")

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}

ds = ds.map(to_text)
print(ds[0]["text"][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=2048,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=3,
        lr_scheduler_type="cosine",
        logging_steps=10,
        output_dir="qlora-out",
        report_to="none",
    ),
)
trainer.train()
model.save_pretrained("student-adapter")
tokenizer.save_pretrained("student-adapter")

In [ ]:
# Sanity check: student generates SQL for a held-out phrasing
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": ds[0]["messages"][0]["content"]},
    {"role": "user", "content": "Show me sales totals broken down by nation"},
]
ids = tokenizer.apply_chat_template(msgs, return_tensors="pt",
                                    add_generation_prompt=True).to("cuda")
out = model.generate(ids, max_new_tokens=256)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## Bake-off

Serve the merged student behind an OpenAI-compatible endpoint (vLLM:
`vllm serve student-merged --port 8000`), then point the agent at it:

```
AI_BASE_URL=http://localhost:8000/v1  AI_API_KEY=local  AI_MODEL=student-merged
python -m src.eval_harness
```

Same golden set, same scoring — record student vs teacher accuracy in
`evals/results.json` and the README table. That is the honest comparison.